In [ ]:
import json

def count_ids(node):
    if isinstance(node, dict):
        return (1 if "id" in node else 0) + sum(count_ids(v) for v in node.values())
    if isinstance(node, list):
        return sum(count_ids(item) for item in node)
    return 0

with open("Crop_Disease_train_qwenvl.json", "r") as f:
    data = json.load(f)
    print(count_ids(data))


145261


In [ ]:
import os
import json
import csv
from google import genai
from google.genai import types
from google.genai.errors import APIError
from tqdm import tqdm
from itertools import islice

INPUT_JSON = "Crop_Disease_train_qwenvl.json"
OUT_DIR = "processed_cddm"
os.makedirs(OUT_DIR, exist_ok=True)
CSV_FILE = os.path.join(OUT_DIR, "cddm_classifier.csv")
JSONL_FILE = os.path.join(OUT_DIR, "cddm_rag.jsonl")

BATCH_SIZE = 10
MODEL_NAME = "gemini-2.5-flash"
CSV_FIELDS = ["id", "image_path", "crop", "disease", "remedy"]

try:
    client = genai.Client()
except Exception as e:
    print(f"Error initializing Gemini client: {e}")
    client = None 

def make_batch_prompt(strands):

    strands_json = json.dumps(strands, ensure_ascii=False, indent=2)

    return f"""
You are a strict data parsing expert. Extract information ONLY from what is explicitly written.

Given a list of conversation strands about crop diseases, you must process ALL of them.
For each strand, extract the following fields:
- id: The tracking ID for the strand (integer provided in the input).
- image_path: The path to the image extracted from the <img> HTML tag in the conversation. If no image is present or the path cannot be determined from the text, set this to an empty string "".
- crop: The type of crop (e.g., "Apple"). If not explicitly stated in the text, set this to "".
- disease: The name of the disease (e.g., "Apple scab"). CRITICAL: If the text indicates the plant is healthy, has no disease, or no issues are mentioned, set this to "healthy". If not explicitly stated in the text, set this to "".
- remedy: The treatment or management instructions for the disease. CRITICAL: This field MUST remain an empty string "" unless a remedy, treatment, or management solution is EXPLICITLY mentioned in the text. Do NOT infer, suggest, or generate remedies.

STRICT REQUIREMENTS:
- NEVER infer, guess, or hallucinate values. Only use information explicitly present in the strand text.
- For remedy: ONLY extract if treatment/remedy/solution is explicitly mentioned. If not mentioned, MUST be empty string "".
- For disease: If the plant is described as healthy or no disease is mentioned, set to "healthy".
- If a value is unknown, missing, or ambiguous, use the empty string "" (except id, which must be the provided integer).
- Output MUST be a single JSON array with exactly one JSON object for each input strand, in the same order as provided.
- Each object in the array must contain exactly the keys: "id", "image_path", "crop", "disease", and "remedy". No other keys are allowed.
- Do not include any other text, explanations, or markdown formatting (like ```json) before or after the JSON array.

Input Conversation Strands (JSON Array):
{strands_json}
"""

def parse_batch_with_gemini(strands):
    if not client:
        return [{"id": s['id'], "error": "Gemini client not initialized"} for s in strands]

    prompt = make_batch_prompt(strands)
    batch_ids = [s['id'] for s in strands]

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=[prompt],
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema={
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "id": {"type": "integer"},
                            "image_path": {"type": "string"},
                            "crop": {"type": "string"},
                            "disease": {"type": "string"},
                            "remedy": {"type": "string"},
                        },
                        "required": ["id", "image_path", "crop", "disease", "remedy"]
                    }
                }
            )
        )

        if response.text:
            return json.loads(response.text)
        else:
            print(f"Warning: Gemini returned an empty response for batch {batch_ids}.")
            return [{"id": _id, "error": "Empty API response."} for _id in batch_ids]

    except APIError as e:
        print(f"Gemini API Error for batch {batch_ids}: {e}")
        return [{"id": _id, "error": f"API Error: {e}"} for _id in batch_ids]
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON response for batch {batch_ids}: {e}")
        return [{"id": _id, "error": f"JSON Decode Error: {e}"} for _id in batch_ids]
    except Exception as e:
        print(f"An unexpected error occurred for batch {batch_ids}: {e}")
        return [{"id": _id, "error": str(e)} for _id in batch_ids]



data = json.load(f)

for i, strand in enumerate(data):
    strand['id'] = i + 1

total_strands = len(data)
success_count = 0

write_header = not os.path.exists(CSV_FILE) or os.stat(CSV_FILE).st_size == 0

with open(CSV_FILE, "a", newline="", encoding="utf-8") as csvfile, \
        open(JSONL_FILE, "a", encoding="utf-8") as jsonlfile:

    csv_writer = csv.DictWriter(csvfile, fieldnames=CSV_FIELDS)
    if write_header:
        csv_writer.writeheader()
    
    data_iterator = iter(data)
    batches_processed = 0
    
    with tqdm(total=total_strands, desc="Processing strands with Gemini") as pbar:
        while True:
            batch = list(islice(data_iterator, BATCH_SIZE))
            if not batch:
                break 

            results = parse_batch_with_gemini(batch)
            
            for r in results:
                if isinstance(r, dict) and "error" not in r:
                    csv_writer.writerow(r)
                    rag_entry = {k: v for k, v in r.items() if k != 'error'}
                    json.dump(rag_entry, jsonlfile, ensure_ascii=False)
                    jsonlfile.write("\n")
                    success_count += 1
            
            csvfile.flush()
            jsonlfile.flush()
            
            pbar.update(len(batch))
            batches_processed += 1
            
print("\n--- Summary ---")
print(f"Total strands requested: {total_strands}")
print(f"Successfully parsed entries (CSV/JSONL): {success_count}")
print(f"Estimated failed entries (or not yet run): {total_strands - success_count}")
print(f"CSV saved to {CSV_FILE}")
print(f"JSONL saved to {JSONL_FILE}")

Processing strands with Gemini:   0%|          | 110/145261 [04:39<102:22:22,  2.54s/it]


KeyboardInterrupt: 